In [ ]:
!python -m pip install -r requirements.txt

In [ ]:
import re #regex
import pandas as pd #pandas
import random #to shuffle the datasets
from sklearn.feature_extraction.text import CountVectorizer #changes the text into numbers
from sklearn.model_selection import train_test_split #splits the data into training and testing data
from sklearn.linear_model import LogisticRegression #uses multiclass logistic regression
from sklearn.metrics import classification_report, accuracy_score #imports evaluator for logistic regression
import pickle #to save the model
import torch # provides tensors (special arrays) and does hard math calculations
from time import time #keeps track of time
from torch.utils.data import Dataset, DataLoader #tells python where the data is and loads it quickly
from transformers import AutoTokenizer, AutoModelForSequenceClassification # the model distillbert and a translator so it understands text
from torch.optim import AdamW #optimizer, looks at what can be improved
from tqdm import tqdm #shows a progress bar

#get access to the data
df = pd.read_csv("Option1-TrainingDataset.csv")


#Variables

#emotion labels to make it clearer in the report than using the numbers
emotion_labels = ['Anger (0)', 'Fear (1)', 'Joy (2)', 'Love (3)', 'Sadness (4)', 'Surprise (5)']
number_of_labels = len(emotion_labels)

#pre-trained model
model_name = "distilbert-base-uncased"
batch_size = 16 #Processes tweets in sets of 16
epochs = 3 # does 3 full runs of the dataset
learning_rate = 5e-5 #how fast it learns

# Tries to use GPU but switches to CPU if not
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("\n")


#Preprocessing

def clean_text(text):
    if not isinstance(text, str): #if not a string replace with empty string
        text = ''

    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Remove links
    text = re.sub(r'#', '', text) #Remove hashtags but keeps word after it
    text = re.sub(r'[^\w\s]', '', text) # Removes punctuation
    return text

#Pytorch emotiondataset class

class EmotionDataset(Dataset):
    #creates the class
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
      #changes the data into tensors for the gpu to understand
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    #how much stuff is inside
    def __len__(self):
        return len(self.labels)


#runs the clean text function
df['text_cleaned'] = df['text'].astype(str).apply(clean_text)

#assigns variables for the columns
texts = df['text_cleaned'].values
labels = df['label'].values

# split the data between being used for training and testing
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


# the models translator so it understands inputs
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Training and testing data split and model translator loaded")


# Converts the text to be understandable by model. Truncates if too long. Adds padding if too short.
train_encodings = tokenizer(
    list(X_train_text),
    truncation=True,
    padding=True
)
test_encodings = tokenizer(
    list(X_test_text),
    truncation=True,
    padding=True
)


#creates the objects
train_dataset = EmotionDataset(train_encodings, list(y_train))
test_dataset = EmotionDataset(test_encodings, list(y_test))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
print("Objects created. Ready to train")
print("\n")


# Fine tuning the model

# gets the model and makes it ready for multiclass classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=number_of_labels
)
model.to(device)
# Use AdamW optimizer to change weights to be better
optimizer = AdamW(model.parameters(), lr=learning_rate)

print(f"Fine tuning {model_name} for {epochs} epochs.")

#runs the training process
for epoch in range(epochs):
    model.train() # Set model to training mode
    start_time = time()

    #starts learning, tqdm does a progress bar to show
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad() #gets rid of old data

        # Moves stuff to the device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Makes a guess based on the score for each emotion
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss #calculates how many were wrong in the batch

        #based on the ones that were wrong uses that data to optimise and change the gradients and weights
        loss.backward()
        optimizer.step()

    end_time = time()
    print(f"Epoch {epoch+1} Complete. Loss: {loss.item():.4f} | Time: {end_time - start_time:.2f}s")
    print("Fine tuning done")
print("\n")

#Evaluation

print("Evaluating")
model.eval()
all_preds = []
all_labels = []

#doesnt need gradients for testing, runs faster
with torch.no_grad():
    #loads testing data
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        outputs = model(input_ids, attention_mask=attention_mask)
        # Tries to predict which emotion it would be
        predictions = torch.argmax(outputs.logits, dim=-1)

        #changes from tensors to numpy to be used for calculations
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

#Creates report showing precision, recall, and f1-score for the emotions
print("Classification Report (for the 6 emotions):")
print(classification_report(all_labels, all_preds, target_names=emotion_labels))

# Saving the model
model.save_pretrained("distilbert_emotion_model")
tokenizer.save_pretrained("distilbert_emotion_model")
print("Saved  in 'distilbert_emotion_model' directory.")

c:\Users\olive\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu




c:\Users\olive\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\olive\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Training and testing data split and model translator loaded
Objects created. Ready to train




Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch 1: 100%|██████████| 800/800 [3:02:49<00:00, 13.71s/it]      


Epoch 1 Complete. Loss: 0.1308 | Time: 10969.51s
Fine tuning done


Epoch 2: 100%|██████████| 800/800 [32:52<00:00,  2.47s/it]


Epoch 2 Complete. Loss: 0.0846 | Time: 1972.31s
Fine tuning done


Epoch 3:  46%|████▋     | 371/800 [15:12<12:22,  1.73s/it]